# Weekly Music History Aggregator

This notebook aggregates historical music information on a temporal timeline and lists new releases with artist, label, album, and track name.

What it does:
- Accepts a target date or date range
- Aggregates historical releases from that week across past years on a timeline
- Fetches current new releases from authoritative indie/alt sources with artist, label, album, and track details
- Outputs a clean summary for DJ planning

Designed for new programmers with full documentation and inline comments.

## 0. Install required packages

Install any required Python packages before running the notebook.

In [ ]:
import subprocess
import sys

required_packages = [
    'requests',
    'beautifulsoup4',
    'lxml'
]

for package in required_packages:
    try:
        __import__(package.replace('-', '_'))
        print(f'{package} already installed')
    except ImportError:
        print(f'Installing {package}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
        print(f'{package} installed successfully')

print('All required packages are ready')

## 1. Setup – imports and configuration

Import libraries and set up API keys and configuration. The notebook uses environment variables for API keys with fallbacks.

In [ ]:
import os  # for environment variables
import json  # for saving/loading data
import requests  # for API calls
from datetime import datetime, timedelta  # for date handling
from collections import Counter, defaultdict  # for counting and grouping

# --- Configuration ---
# API keys from environment, with fallbacks for testing
KZSU_API_KEY = os.getenv('KZSU_LIBRARY_API_KEY', 'b3fcd72414bde44530b5fd7b585aaafc08ca41b1')
HEADERS = {'X-APIKEY': KZSU_API_KEY, 'Content-Type': 'application/json'}

# Base URLs
BASE_URL_V2 = 'https://zookeeper.stanford.edu/api/v2/'
BASE_URL_V1 = 'https://zookeeper.stanford.edu/api/v1/'

# Output directory for show scripts
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. User-customizable parameters

Set the target date, date range, and show length. These are the main controls for the DJ.

In [ ]:
# Target date for music history week. Format: YYYY-MM-DD
TARGET_DATE = '2026-09-08'  # <-- change this to any date

# Optional date range. If set, use start and end dates instead of single date
DATE_RANGE_START = None  # e.g., '2025-09-01'
DATE_RANGE_END = None    # e.g., '2025-09-07'

# Show length in minutes
SHOW_LENGTH_MINUTES = 120  # <-- 120 for 2 hours, 180 for 3 hours, etc.

# DJ info
DJ_AIRNAME = 'Stace'  # <-- your airname
SHOW_NAME = 'Library Show'

print(f"Building show for {TARGET_DATE}, length {SHOW_LENGTH_MINUTES} minutes, DJ {DJ_AIRNAME}")

## 3. Helper functions

Functions to fetch data from KZSU API and external sources.

In [ ]:
def get_historical_releases_for_week(target_date):
    """
    Find releases that came out on the same month/day in previous years.
    This builds the 'music history' segment.
    """
    # Parse target date
    target_dt = datetime.fromisoformat(target_date)
    month_day = target_dt.strftime('%m-%d')
    
    # Look back 10 years for releases on this month/day
    releases = []
    for year_offset in range(1, 11):
        year = target_dt.year - year_offset
        # Query KZSU API for albums released on this month/day
        # Note: This is a simplified example; actual API may need different filters
        url = f'{BASE_URL_V1}album'
params = {'filter[release_date]': f'{year}-{month_day}'}
        try:
            response = requests.get(url, headers=HEADERS, params=params, timeout=10)
            response.raise_for_status()
            data = response.json().get('data', [])
            for album in data:
                attrs = album.get('attributes', {})
                releases.append({
                    'artist': attrs.get('artist'),
                    'album': attrs.get('album'),
                    'release_date': f'{year}-{month_day}',
                    'year': year
                })
        except Exception as e:
            print(f"Error fetching releases for {year}-{month_day}: {e}")
    return releases

def fetch_new_releases():
    """
    Fetch new indie/alt releases from authoritative sources.
    This is a placeholder for real API integrations with Bandcamp, Pitchfork, etc.
    """
    # In production, this would call:
    # - Bandcamp API / RSS feeds
    # - Pitchfork reviews API
    # - IndieIsNotAGenre RSS
    # For now, return mock data
    mock_releases = [
        {'artist': 'Example Band', 'album': 'New Album', 'source': 'Pitchfork', 'url': 'https://pitchfork.com'},
        {'artist': 'Indie Artist', 'album': 'Fresh Sounds', 'source': 'Bandcamp', 'url': 'https://bandcamp.com'},
        {'artist': 'Alt Group', 'album': 'Latest EP', 'source': 'IndieIsNotAGenre', 'url': 'https://indieisnotagenre.com'}
    ]
    return mock_releases

## 4. Build music history profile

Gather historical releases for the target week and build a profile.

In [ ]:
print('[1] Building music history profile...')
historical_releases = get_historical_releases_for_week(TARGET_DATE)
print(f'Found {len(historical_releases)} historical releases')

# Group by year for display
releases_by_year = defaultdict(list)
for r in historical_releases:
    releases_by_year[r['year']].append(r)

# Print summary
for year in sorted(releases_by_year.keys(), reverse=True):
    print(f"{year}: {len(releases_by_year[year])} releases")

## 5. Fetch new release suggestions

Get current new releases from indie/alt sources.

In [ ]:
print('[2] Fetching new release suggestions...')
new_releases = fetch_new_releases()
print(f'Found {len(new_releases)} new releases')
for r in new_releases:
    print(f"{r['artist']} - {r['album']} ({r['source']})")

## 6. Build show script

Construct the markdown show script with track listings, LIDs, airbreaks, PSAs, promos, and ticket giveaways.

In [ ]:
def build_show_script():
    """Build complete markdown show script."""
    lines = []
    
    # Header
    lines.append(f'# {SHOW_NAME} - Show Script')
    lines.append(f'DJ: {DJ_AIRNAME}')
    lines.append(f'Date: {TARGET_DATE}')
    lines.append(f'Show Length: {SHOW_LENGTH_MINUTES} minutes')
    lines.append('')
    
    # Music history intro
    lines.append('## Music History Segment')
    lines.append(f'This week in music history ({TARGET_DATE}):')
    for year in sorted(releases_by_year.keys(), reverse=True)[:5]:
        lines.append(f'
### {year}')
        for r in releases_by_year[year][:3]:
            lines.append(f"- {r['artist']} - {r['album']} (Released {r['release_date']})")
    lines.append('')
    
    # New releases
    lines.append('## New Releases')
    for r in new_releases:
        lines.append(f"- {r['artist']} - {r['album']} [{r['source']}]({r['url']})")
    lines.append('')
    
    # Show timeline
    lines.append('## Show Timeline')
    hours = SHOW_LENGTH_MINUTES // 60
    for hour in range(1, hours + 1):
        start_min = (hour - 1) * 60
        end_min = hour * 60
        lines.append(f'
### Hour {hour} ({start_min}-{end_min} min)')
        lines.append('- **Legal ID**: KZSU Stanford')
        lines.append('- **Airbreak**: Station ID and music')
        lines.append('- **PSA 1**: Community message')
        lines.append('- **PSA 2**: Public service announcement')
        lines.append('- **PROMO**: Show promotion')
        if hour == 2:
            lines.append('- **TICKET GIVEAWAY**: Announce ticket giveaway!')
        lines.append('- Tracks:')
        # Placeholder for actual track listings
        lines.append('  - Artist - Track Title (Album)')
    
    return '\n'.join(lines)

script = build_show_script()
print(script[:1000])  # Print first 1000 chars

## 7. Save show script

Write the show script to a markdown file.

In [ ]:
script_filename = f'show_script_{TARGET_DATE}_{DJ_AIRNAME}.md'
script_path = os.path.join(OUTPUT_DIR, script_filename)
with open(script_path, 'w') as f:
    f.write(script)
print(f'Show script saved to {script_path}')

## Notes for new programmers

- Change `TARGET_DATE`, `SHOW_LENGTH_MINUTES`, and `DJ_AIRNAME` at top to customize
- `get_historical_releases_for_week` queries KZSU API for albums released on same month/day in past years
- `fetch_new_releases` is a placeholder; replace with real API calls to Bandcamp, Pitchfork, etc.
- Show script includes hourly LID, airbreaks, 2 PSAs, 1 PROMO per hour, and ticket giveaway in hour 2
- Output is markdown formatted for easy reading and printing